# İnsan-Döngüsünde: Ön-Eylem Kapıları, Risk Sınıflandırması ve Denetim Kaydı

Bu dersin README dosyası, ajan yanıtı oluşturduktan sonra kullanıcıdan `ONAYLA` veya `REDDET` istenen kısa bir snippet ile İnsan-Döngüsü (Human-in-the-Loop) kavramını tanıtıyor. Bu desen iyi bir başlangıç noktasıdır, ancak üretim ortamındaki HITL uygulamaları yaygın olarak üç ek öğeye ihtiyaç duyar:

1. Ajana riskli bir adımı yürütmeden **önce** çalışan bir **ön-eylem kapısı**, böylece maliyet, geri dönülemezlik ve gecikme kontrol altında tutulur.
2. Düşük riskli işlemlerin otomatik yürütülmesi, orta riskli işlemlerin toplu onaylanması ve yalnızca yüksek riskli işlemlerin bir insanda bloklanması için **risk sınıflandırması**.
3. Her kapı kararının JSONL olarak kaydedildiği ve bir red durumunda sadece `Revize ediliyor...` yazdırmak yerine ajanı yapılandırılmış bir nedenle yeniden yönlendiren bir **denetim kaydı ve revizyon döngüsü**.

Bu notebook, her birini `06-system-message-framework.ipynb` ile aynı ilkelere dayanarak oluşturur. `DEMO_MODE = True` iken sonuna kadar çalışır (etkileşimli giriş gerekmez) veya `DEMO_MODE = False` iken gerçek `input()` istemleriyle çalışır. Not: DEMO_MODE'da üçüncü hedefin yeniden deneme kısmı senaryolaştırılmıştır, böylece döngü mekanikleri baştan sona görünür. Gerçek revizyon odaklı yeniden sınıflandırma için `DEMO_MODE = False` ve bir operatör gerekir.

**Kapsam dışı (diğer derslerde ele alınır):** kimlik doğrulama ve erişim kontrolü (Ders 06 README tehdit 2), araç çağırma ara yazılımı (Ders 14 MAF derin inceleme), çok ajanlı tartışma desenleri.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Desen 1: Ön işlem kapısı

README'deki HITL kod parçası önce ajanı çağırır, sonra kullanıcıdan çıktıyı onaylamasını ister. Bu bir **son işlem** akışıdır. Ajan zaten çalışmıştır, yani LLM çağrısı maliyeti zaten ödenmiştir ve herhangi bir yan etki (gönderilen e-posta, yazılan veri tabanı satırı, paylaşılan yorum) zaten gerçekleşmiştir.

Bir **ön işlem** akışı, ajan riskli adımı çalıştırmadan önce kapıyı ekler. Ajan işlemi önerir, kapı çalıştırmaya karar verir ve sadece onayda yan etki meydana gelir.

| Kapsam | Son işlem onayı (README kodu) | Ön işlem kapısı (bu not defteri) |
|---|---|---|
| Onay ne zaman gerçekleşir? | Ajan çıktı ürettikten sonra | Herhangi bir yan etki gerçekleşmeden önce |
| Reddedildiğinde LLM maliyeti | Zaten ödenmiştir | Sadece öneri için ödenir, işlem için değil |
| Geri alınamaz yan etkiler | Mümkün (işlem zaten gerçekleşmiştir) | Engellenmiştir |
| Denetim netliği | Onay bir yazdırma ifadesidir | Onay, zaman damgası, işlem, neden içeren bir JSON kaydıdır |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Desen 2: Risk seviyelendirmesi

Her eylemin insan onayı gerektirmesi gerekmez. Halka açık bir API üzerinden salt okunur bir sorgulamanın riski, müşteriye e-posta göndermekten farklıdır. İkisini aynı şekilde ele almak operatörün dikkatinin boşa harcanmasına ve ajanın yavaşlamasına neden olur.

Basit bir 3 seviyeli model:

| Seviye | Örnekler | Onay akışı |
|---|---|---|
| `düşük` (salt okunur) | Bir bilgi tabanında arama yapmak, uçuş seçeneklerini incelemek, halka açık bir web sayfasını getirmek | Otomatik yürütme, denetim için kayıt altına alınır |
| `orta` (ucuz değişiklik) | Bir sonucu önbelleğe almak, bir sayacı artırmak, bir hatırlatıcı planlamak | Otomatik yürütme, ancak günlük toplu inceleme |
| `yüksek` (dışa dönük veya geri dönüşü olmayan) | E-posta göndermek, karttan ücret almak, halka açık bir kanala gönderi yapmak | İnsan onayı beklenir |

Bu bir seviyelendirmedir. Üretim sistemleri genellikle daha ayrıntılı seviyeler kullanır (örneğin, AWS IAM izin seviyeleri, rol tabanlı erişim seviyeleri). Aşağıdaki 3 seviyeli versiyon, salt okunur ve yan etkisi olan eylemleri karıştıran bir ajan için en küçük kullanışlı versiyondur.

Aşağıdaki sınıflandırıcı, demo deterministik ve ucuz kalsın diye anahtar kelime sezgilerini kullanır. Bir üretim sisteminde bunu öğrenilmiş bir sınıflandırıcı veya politika motoruyla değiştirirsiniz.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Model 3: Denetim kaydı ve revizyon döngüsü

`print("Response approved.")` bir denetim kaydı değildir. Güven için, her kapı kararı daha sonra sorgulayabileceğiniz, yeniden oynatabileceğiniz veya bir olay incelemesine ekleyebileceğiniz yapılandırılmış bir olay olarak kaydedilmelidir.

İki parça:

1. **Sadece eklemeli JSONL.** Karar başına bir satır, zaman damgası, eylem, seviye, karar, neden ile. Grep yapmak kolay, daha sonra gerçek bir günlük deposuna göndermek kolay.
2. **Reddetme halinde revizyon döngüsü.** Kapı `deny` döndürdüğünde, ajan reddetme nedenini bağlamda kendisine yeniden sorar, böylece sonraki öneri problemi önleyebilir.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Ek kaynaklar

Birkaç diğer açık kaynak proje, bu HITL (insan döngüde) desenlerinin varyasyonlarını uygular. Yığınınıza en uygun olanı bulmak için yaklaşımları karşılaştırın:

- **LangChain** insan döngüde araç sarmalayıcıları ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): insan girdisi için yürütmeyi duraklatan araç sarmalayıcıları.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ bunu yeniden yapılandırdı): çok ajanlı konuşmalarda insanı temsil etmek için özel olarak bir ajan rolü kullanır.
- **Microsoft Agent Framework (MAF)** işlev çağrısı arası yazılım katmanı ([docs](https://learn.microsoft.com/agent-framework/)): her araç/fonksiyon çağrısının etrafında çalışan, geçiş mantığı ve onay akışları için uygun bir arası yazılım katmanı.

Her proje üç alt deseni farklı şekilde ele alır: LangChain bunları araç olarak sarar, AutoGen bir ajan rolü kullanır ve Microsoft Agent Framework işlev çağrısı arası yazılım katmanı kullanır. Kendi ajanınız için bir tasarım seçmeden önce bir veya iki uygulamayı baştan sona okuyun.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
